# Data Cleaning und Geospatial Preprocessing

In [1]:
import pandas as pd
import geopandas as gpd 

This stage focuses on preparing the raw dataset for analysis by ensuring consistency, correctness, and relevance of the data. The original reports dataset is loaded and converted into a proper datetime format to enable temporal filtering. Only records from the period 2016 to 2025 are retained to ensure a consistent ten-year study period. Additional temporal features, such as the exact date and year, are extracted to facilitate later time-based analysis. Irrelevant columns are removed to reduce data complexity and improve computational efficiency. Finally, the cleaned dataset is stored as a processed file.

In [ ]:
# Load raw data
reports = pd.read_csv("../data/raw/stzh.zwn_meldungen_p.csv")

# Date conversion
reports["requested_datetime"] = pd.to_datetime(reports["requested_datetime"])

# Filter time range
reports = reports[(reports["requested_datetime"] >= "2016-01-01") & (reports["requested_datetime"] <= "2025-12-31")]

# Create date features
reports["date"] = reports["requested_datetime"].dt.date
reports["year"] = reports["requested_datetime"].dt.year

# Keep only needed columns 
reports = reports[["date", "year", "service_code", "e", "n"]]
display(reports.head())

# Save cleaned table in processed data
reports.to_csv("../data/processed/reports_cleaned.csv", index=False)

,date,year,service_code,e,n
1509,2016-06-27,2016,Brunnen/Hydranten,2683920,1245604
1959,2018-06-29,2018,Strasse/Trottoir/Platz,2682182,1248788
2056,2016-01-20,2016,Beleuchtung/Uhren,2683195,1247605
2139,2021-01-21,2021,Brunnen/Hydranten,2683931,1248842
2266,2023-05-13,2023,Strasse/Trottoir/Platz,2681716,1247084


This preprocessing step prepares all spatial datasets for analysis. The  report data is converted into a GeoDataFrame by transforming coordinate columns into geometric point features and assigning a consistent coordinate reference system (EPSG:2056). District boundaries are loaded, and enriched with area information in square kilometres to enable standardized comparisons across regions. In addition, lake geometries are clipped to the Zurich municipal boundary and filtered to retain only Zürichsee. Finally, all processed geospatial layers are exported as GeoPackage.

In [3]:
# Reports
reports_df = pd.read_csv("../data/processed/reports_cleaned.csv")
reports_gdf = gpd.GeoDataFrame(reports_df, geometry=gpd.points_from_xy(reports_df.e, reports_df.n), crs="EPSG:2056")

# district
district_gdf = gpd.read_file("../data/raw/Zurich_quarters.gpkg").to_crs(2056)
district_gdf["area_km2"] = district_gdf.geometry.area / 1_000_000

# Lakes (Zürichsee extraction)
zh_district = gpd.read_file("../data/raw/seen_stadtZH.gpkg")

lakes = gpd.read_file("../data/raw/AV_Gewasser_-OGD.gpkg").to_crs(2056)

perimeter = zh_district.union_all()
lakes_zh = gpd.clip(lakes, perimeter)

zürichsee_gdf = lakes_zh[lakes_zh["gewaessername"] == "Zürichsee"]

# Save Geodata
reports_gdf.to_file("../data/processed/reports_gdf.gpkg", driver="GPKG")
district_gdf.to_file("../data/processed/district.gpkg", driver="GPKG")
zürichsee_gdf.to_file("../data/processed/zurichsee.gpkg", driver="GPKG")

The overview shows that all columns were loaded correctly and that the data types are consistent across the dataset. The missing value check `(isna().sum())`indicates that there are no missing values in the relevant columns. Therefore, no obvious data gaps or incomplete records remain. Based on these checks, the essential data cleaning process can be considered complete.

In [4]:
# Overview
print(reports_gdf.info())

# Missing Values
print(reports_gdf.isna().sum())

# Data Types
print(reports_gdf.dtypes)

<class 'geopandas.geodataframe.GeoDataFrame'>
RangeIndex: 62245 entries, 0 to 62244
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype   
---  ------        --------------  -----   
 0   date          62245 non-null  str     
 1   year          62245 non-null  int64   
 2   service_code  62245 non-null  str     
 3   e             62245 non-null  int64   
 4   n             62245 non-null  int64   
 5   geometry      62245 non-null  geometry
dtypes: geometry(1), int64(3), str(2)
memory usage: 2.8 MB
None
date            0
year            0
service_code    0
e               0
n               0
geometry        0
dtype: int64
date                 str
year               int64
service_code         str
e                  int64
n                  int64
geometry        geometry
dtype: object
